### Set paths and object name

In [ ]:
obj_name = 'ficus'

gsplat_folder = './'
rt_folder = '/home/operel/Code/deploy/ray-tracing-gaussians/extra_info/'

### Load pickled data

In [ ]:
import pickle
import pandas as pd
try:
    import seaborn
except:
    !pip install seaborn
    import seaborn

In [ ]:
with open(gsplat_folder + f'{obj_name}.pickle', 'rb') as handle:
    gsplats_data = pickle.load(handle)
print(f"Loaded gsplat data for {gsplats_data['extra_dict']['iteration'][-1]} iterations")

with open(rt_folder + f'{obj_name}.pickle', 'rb') as handle:
    rt_data = pickle.load(handle)
print(f"Loaded 3dgrt data for {rt_data['extra_dict']['iteration'][-1]} iterations")

#### Training data

In [ ]:
all_keys = set(gsplats_data['extra_dict'].keys()) | set(rt_data['extra_dict'].keys())
num_gsplat_entries =  len(gsplats_data['extra_dict']['iteration'])
num_rt_entries =  len(rt_data['extra_dict']['iteration'])
training_summary = dict(
    method = ['gsplat'] * num_gsplat_entries + ['3dgrt'] * num_rt_entries
)

for key in all_keys:
    gsplat_entries = gsplats_data['extra_dict'][key] if key in gsplats_data['extra_dict'] else [None] * num_gsplat_entries
    rt_entries = rt_data['extra_dict'][key] if key in rt_data['extra_dict'] else [None] * num_rt_entries
    training_summary[key] = gsplat_entries + rt_entries

training_summary = pd.DataFrame(training_summary)
training_summary

#### Final Gaussians

In [ ]:
all_keys = set(gsplats_data['gaussians'].keys()) | set(rt_data['gaussians'].keys())
num_gsplat_entries =  gsplats_data['gaussians']['pos'].shape[0]
num_rt_entries =  rt_data['gaussians']['pos'].shape[0]
gaussians_summary = dict(
    method = ['gsplat'] * num_gsplat_entries + ['3dgrt'] * num_rt_entries
)

for key in all_keys:
    gsplat_entries = gsplats_data['gaussians'][key] if key in gsplats_data['gaussians'] else [None] * num_gsplat_entries
    rt_entries = rt_data['gaussians'][key] if key in rt_data['gaussians'] else [None] * num_rt_entries
    if key.startswith('sh_deg'):
        gsplat_entries = gsplat_entries.reshape(num_gsplat_entries, -1)
        rt_entries = rt_entries.reshape(num_rt_entries, -1)
    gaussians_summary[key] = gsplat_entries.tolist() + rt_entries.tolist()

gaussians_summary = pd.DataFrame(gaussians_summary)
gaussians_summary

### Plots

In [ ]:
sns.relplot(
    data=training_summary, 
    kind="line",
    x='iteration', y='num_gaussians', hue='method'
)

In [ ]:
sns.relplot(
    data=summary, 
    kind="line",
    x='iteration', y='iter_time', hue='method'
)

In [ ]:
sns.relplot(
    data=summary, 
    kind="line",
    x='iteration', y='opacity_mean', hue='method'
)

In [ ]:
sns.relplot(
    data=summary, 
    kind="line",
    x='iteration', y='features_albedo_mean', hue='method'
)

In [ ]:
sns.relplot(
    data=summary, 
    kind="line",
    x='iteration', y='features_albedo_std', hue='method'
)